# game
## payoff matrix
T,R,P,S = 3,2,1,0

In [ ]:
C, D = 0, 1

PAYOFF = [
    [(2, 2), (0, 3)],
    [(3, 0), (1, 1)]
]

class MatchState:
    def __init__(self, N=5, T=100):
        self.N = N
        self.T = T
        self.reset_match()
    
    def set_strategies(self, s1, s2):
        self.strategy1_name = str(s1)
        self.strategy2_name = str(s2)

    def reset_game(self):
        if self.history:
            self.all_history.append({
                "strategy1": self.strategy1_name,
                "strategy2": self.strategy2_name,
                "history": self.history.copy()
            })
        self.history = []
        self.t = 0
        self.n += 1

    def reset_match(self):
        self.history = []
        self.all_history = []
        self.t = 0
        self.n = 0

    def step(self, a, b):
        r1, r2 = PAYOFF[a][b]
        self.history.append((a, b, r1, r2))
        self.t += 1
        return r1, r2

    def is_game_over(self):
        return self.t >= self.T

    def is_match_over(self):
        return self.n >= self.N
    
    def match_play(self, strategy1, strategy2):
        self.reset_match()
        self.set_strategies(strategy1, strategy2)

        while not self.is_match_over():
            self.reset_game()
            strategy1.reset()
            strategy2.reset()

            while not self.is_game_over():
                a = strategy1.act(self)
                b = strategy2.act(self)
                self.step(a, b)

        # save last game
        if self.history:
            self.all_history.append({
                "strategy1": self.strategy1_name,
                "strategy2": self.strategy2_name,
                "history": self.history.copy()
            })

        return self.all_history


## Population initiation



In [2]:
class Strategy:
    """
    Base class for IPD strategies.

    Contract:
    - act(state) must return 0 (C) or 1 (D)
    - reset() must clear internal state between matches
    """
    def __str__(self):
        return self.__class__.__name__

    COOP = 0
    DEFECT = 1

    def reset(self):
        """Reset internal state before a new match."""
        pass

    def act(self, state):
        """
        Decide next action.

        Parameters:
            state:
                state.history -> list of (my_action, opp_action)
                state.turn   -> int

        Returns:
            int: 0 (COOP) or 1 (DEFECT)
        """
        raise NotImplementedError

In [3]:
class TitForTat(Strategy):
    def act(self, state):
        if not state.history:
            return self.COOP
        return state.history[-1][1]  # opponent's last move
    
class TitForTwoTats(Strategy):
    def act(self, state):
        if len(state.history) < 2:
            return self.COOP

        last_two = state.history[-2:]
        if last_two[0][1] == self.DEFECT and last_two[1][1] == self.DEFECT:
            return self.DEFECT

        return self.COOP
    
class GrimTrigger(Strategy):
    def reset(self):
        self.triggered = False

    def act(self, state):
        if self.triggered:
            return self.DEFECT

        # check if opponent ever defected
        if any(o == self.DEFECT for _, o, _, _ in state.history):
            self.triggered = True
            return self.DEFECT

        return self.COOP

class Pavlov(Strategy):
    def act(self, state):
        if not state.history:
            return self.COOP

        my_last, opp_last, r_self, _ = state.history[-1]

        # if last outcome was "good" → repeat
        if r_self >= 2:  # CC or DC
            return my_last
        else:
            return 1 - my_last

In [4]:
# state = MatchState(N=5, T=100)
# games = state.match_play(TitForTwoTats(), Pavlov())

# print(games)

In [ ]:
results = {}
strategy_classes = [TitForTat, TitForTwoTats, GrimTrigger, Pavlov]
for S1 in strategy_classes:
    for S2 in strategy_classes:
        s1, s2 = S1(), S2()
        state = MatchState(N=5, T=100)

        games = state.match_play(s1, s2)
        print(f"{S1.__name__} vs {S2.__name__}: {len(games)} games")

TitForTat vs TitForTat: 5 games
TitForTat vs TitForTwoTats: 5 games
TitForTat vs GrimTrigger: 5 games
TitForTat vs Pavlov: 5 games
TitForTwoTats vs TitForTat: 5 games
TitForTwoTats vs TitForTwoTats: 5 games
TitForTwoTats vs GrimTrigger: 5 games
TitForTwoTats vs Pavlov: 5 games
GrimTrigger vs TitForTat: 5 games
GrimTrigger vs TitForTwoTats: 5 games
GrimTrigger vs GrimTrigger: 5 games
GrimTrigger vs Pavlov: 5 games
Pavlov vs TitForTat: 5 games
Pavlov vs TitForTwoTats: 5 games
Pavlov vs GrimTrigger: 5 games
Pavlov vs Pavlov: 5 games
